# Tasneem – Phi-3-mini-4k-instruct Evaluation
Evaluating Microsoft Phi-3-mini on medical open-ended and MCQ tasks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
OUT_DIR      = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Tasneem"
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/questions_w_answers.jsonl"
MCQ_NO_ANS   = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_without_Correct_Answer.csv"
MCQ_ANS      = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_with_Correct_Answer.csv"

MODEL_TAG    = "phi3"

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json, os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── Open questions (lines 100-116) ─────────────────────────────────────
open_qs = []
with open(DATASET_PATH) as fh:
    for line_no, line in enumerate(fh):
        if line_no < 116:
            continue
        if line_no > 133:
            break
        obj = json.loads(line)
        open_qs.append({
            "id":        line_no + 1,
            "question":  obj["Question"],
            "must_have": obj["Must_have"]
        })

print(f"{len(open_qs)} open questions loaded")

In [ ]:
mcq_q = pd.read_csv(MCQ_NO_ANS)
mcq_a = pd.read_csv(MCQ_ANS)
print(f"{len(mcq_q)} MCQ questions loaded")

In [ ]:
# ── Utility functions ────────────────────────────────────────────────────
def extract_answer_letter(output: str):
    out = output.upper()
    for letter in "ABCDEF":
        for marker in (f"{letter})", f"{letter}.",
                       f"ANSWER: {letter}", f"ANSWER:{letter}",
                       f" {letter} ", f"({letter})"):
            if marker in out:
                return letter
    return None


def score_open(response: str, required_terms: list) -> float:
    if not required_terms:
        return 0.0
    low = response.lower()
    matched = 0
    for term in required_terms:
        tokens = term.lower().split()
        threshold = max(1, len(tokens) // 2)
        if sum(t in low for t in tokens) >= threshold:
            matched += 1
    return matched / len(required_terms)

In [ ]:
# ── Load Phi-3-mini ──────────────────────────────────────────────────────
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant,
    device_map="auto", trust_remote_code=True
)
print("Phi-3-mini loaded")

In [ ]:
# ── Open-ended inference ─────────────────────────────────────────────────
records_open = []

for item in open_qs:
    chat = [
        {"role": "system", "content": "You are a precise medical assistant."},
        {"role": "user",   "content": item["question"]}
    ]
    text   = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)

    answer = tokenizer.decode(ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    s      = score_open(answer, item["must_have"])
    records_open.append({"id": item["id"], "question": item["question"],
                         "answer": answer, "must_have_score": s})

open_df = pd.DataFrame(records_open)
open_df.to_csv(f"{OUT_DIR}/phi3_open_tasneem.csv", index=False)
print("Open done. Mean:", round(open_df.must_have_score.mean(), 4))

In [ ]:
# ── MCQ inference ────────────────────────────────────────────────────────
records_mcq = []

for idx, row in mcq_q.iterrows():
    q_text = (
        f"{row['Question_text']}\n"
        f"A) {row['(A)']}\nB) {row['(B)']}\nC) {row['(C)']}\n"
        f"D) {row['(D)']}\nE) {row['(E)']}\nF) {row['(F)']}\n"
        f"\nRespond with a single letter only."
    )
    chat = [
        {"role": "system", "content": "You answer medical MCQs with a single letter."},
        {"role": "user",   "content": q_text}
    ]
    text   = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    reply = tokenizer.decode(ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip().upper()
    pred  = extract_answer_letter(reply) or (reply[-1] if reply else None)
    gold  = mcq_a.loc[idx, "Correct_answer"].strip().upper()

    records_mcq.append({"question": row["Question_text"],
                        f"{MODEL_TAG}_prediction": pred,
                        "correct": gold, "score": pred == gold})

mcq_df = pd.DataFrame(records_mcq)
mcq_df.to_csv(f"{OUT_DIR}/phi3_mcq_tasneem.csv", index=False)
print("MCQ accuracy:", round(mcq_df.score.mean(), 4))

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
row = {
    "model":              MODEL_TAG,
    "mcq_accuracy":       mcq_df.score.mean(),
    "mcq_correct":        int(mcq_df.score.sum()),
    "mcq_total":          len(mcq_df),
    "open_mean":          open_df.must_have_score.mean(),
    "open_median":        open_df.must_have_score.median(),
    "open_std":           open_df.must_have_score.std(),
    "open_min":           open_df.must_have_score.min(),
    "open_max":           open_df.must_have_score.max(),
    "open_good_0.5":      int((open_df.must_have_score >= 0.5).sum()),
    "open_excellent_0.8": int((open_df.must_have_score >= 0.8).sum()),
}
summary_df = pd.DataFrame([row])
summary_df.to_csv(f"{OUT_DIR}/phi3_summary.csv", index=False)
print(summary_df.T)

In [ ]:
import gc
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Memory released")